# Fine-Tune InternVL2-4B on Cantopop Lyrics Corpus

This notebook fine-tunes **OpenGVLab/InternVL2-4B** using **LoRA (QLoRA)** on the `cantopop_corpus_final_583_yue.csv` dataset for Cantonese lyrics generation.

**Target task:** Given a scene/image description, generate structured Cantonese lyrics in YuE format (`[verse]` / `[chorus]`).

> ⚠️ Run this on **Google Colab with a T4/A100 GPU**. Go to `Runtime → Change runtime type → GPU`.


In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB" if torch.cuda.is_available() else "")

CUDA available: True
GPU name: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


## 1. Install Required Libraries

In [2]:
# Pin transformers to 4.44.2 — the version InternVL2-4B was built and tested against.
# Too old (<4.41): missing EncoderDecoderCache
# Too new (>=4.48): all_tied_weights_keys / longrope incompatibilities
!pip install -q "transformers==4.44.2" datasets peft trl accelerate bitsandbytes sentencepiece einops torchvision "torchao>=0.16.0"

# Install pre-built flash-attn wheel (~30 sec) instead of compiling from source (~15 min)
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4/flash_attn-2.7.4+cu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("flash-attn installed from pre-built wheel")
else:
    print("Pre-built wheel failed (likely wrong CUDA/Python version), skipping flash-attn")
    print(result.stderr[-500:])

import transformers
print("transformers version:", transformers.__version__)


Pre-built wheel failed (likely wrong CUDA/Python version), skipping flash-attn
ERROR: flash_attn-2.7.4+cu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl is not a supported wheel on this platform.

transformers version: 5.7.0


## 2. Clone Repo from GitHub & Load CSV

The CSV is already in the `second_model` branch of the GitHub repo — we clone it directly, no Drive upload needed.

In [3]:
import os
import pandas as pd

# Clone the repo (second_model branch) if not already cloned
REPO_URL = "https://github.com/Chr1sNga1/Image2CantonSong.git"
REPO_DIR = "/content/Image2CantonSong"

if not os.path.exists(REPO_DIR):
    !git clone --branch second_model --single-branch {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

CSV_PATH = f"{REPO_DIR}/cantopop_corpus_final_583_yue.csv"

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows")
print(df.columns.tolist())
df.head(3)

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 930 bytes | 465.00 KiB/s, done.
From https://github.com/Chr1sNga1/Image2CantonSong
   aaad9a5..d2b6bc8  second_model -> origin/second_model
Updating aaad9a5..d2b6bc8
Fast-forward
 finetune_internvl2_4b_cantopop.ipynb | 69 ++++++++++++++++--------------------
 1 file changed, 30 insertions(+), 39 deletions(-)
Loaded 583 rows
['Artist', 'Title', 'Lyricist', 'Lyrics_Raw', 'Released on', 'Genius_ID', 'Lyrics_YuE']


,Artist,Title,Lyricist,Lyrics_Raw,Released on,Genius_ID,Lyrics_YuE
0,張國榮 (Leslie Cheung),追 (Chase),林夕,[張國榮《追》歌詞]\n\n[主歌一]\n這一生 也在進取\n這分鐘 卻掛念誰\n我會說 是...,1994,6658743,[Verse]\n這一生 也在進取\n這分鐘 卻掛念誰\n我會說 是唯獨你不可失去\n好風光...
1,張國榮 (Leslie Cheung),當年情 (The Sentimental Past),黃霑,[張國榮《當年情》歌詞]\n\n[主歌一]\n輕輕笑聲 在為我送溫暖\n你為我注入快樂強電\...,"October 1, 1986",9983765,[Verse]\n輕輕笑聲 在為我送溫暖\n你為我注入快樂強電\n輕輕說聲 漫長路快要走過\...
2,張國榮 (Leslie Cheung),最愛是誰 My Dearest,潘源良,[張國榮「最愛是誰 My Dearest」歌詞]\n\n[主歌一]\n在世間尋覓愛侶 尋獲了...,"March 17, 2023",8934507,[Verse]\n在世間尋覓愛侶 尋獲了但求共聚\n然而共處半生都過去 我偏偏又後悔\n別了...


## 3. Preprocess & Tokenize Dataset

We format each row as a **prompt → response** pair:
- **Prompt:** A fixed generic instruction to write quality structured Cantonese lyrics — no artist/title metadata, so the model learns *how good lyrics look*, not a specific style.
- **Response:** The `Lyrics_YuE` column (already structured with `[verse]`/`[chorus]`)

In [4]:
MODEL_ID = "OpenGVLab/InternVL2-4B"
MAX_LENGTH = 1024  # adjust if you hit OOM

# Fixed generic instruction — no artist/metadata
# The model learns quality & [verse]/[chorus] structure from the corpus itself
INSTRUCTION = (
    "你是一個粵語流行歌詞創作助手。\n"
    "請創作一首優質的粵語歌詞，使用繁體中文，分為 [verse] 和 [chorus]。\n"
    "歌詞應情感豐富、押韻自然，適合香港粵語演唱。\n"
    "請直接輸出歌詞，不要有任何額外說明。"
)

def build_sample(row):
    lyrics = str(row.get("Lyrics_YuE", "")).strip()
    # InternVL2 chat template format
    return f"<|im_start|>user\n{INSTRUCTION}<|im_end|>\n<|im_start|>assistant\n{lyrics}<|im_end|>"

# Drop rows with empty lyrics
df = df.dropna(subset=["Lyrics_YuE"])
df = df[df["Lyrics_YuE"].str.strip().str.len() > 50].reset_index(drop=True)
print(f"Clean samples: {len(df)}")

samples = [build_sample(row) for _, row in df.iterrows()]
print("Example sample:\n")
print(samples[0][:600])

Clean samples: 583
Example sample:

<|im_start|>user
你是一個粵語流行歌詞創作助手。
請創作一首優質的粵語歌詞，使用繁體中文，分為 [verse] 和 [chorus]。
歌詞應情感豐富、押韻自然，適合香港粵語演唱。
請直接輸出歌詞，不要有任何額外說明。<|im_end|>
<|im_start|>assistant
[Verse]
這一生 也在進取
這分鐘 卻掛念誰
我會說 是唯獨你不可失去
好風光 似幻似虛
誰明人生樂趣？
我會說 為情為愛
仍然是對

[Pre-Chorus]
誰比你重要？
成功了敗了也完全無重要
誰比你重要？
狂風與暴雨都因你燃燒

[Chorus]
一追再追
只想追趕生命裡一分一秒 (一分一秒)
原來多麼可笑
你是真正目標
一追再追
追蹤一些生活最基本需要 (基本需要)
原來早不缺少 Woah
有了你 即使平凡卻最重要

[Verse]
好光陰 縱沒太多
一分鐘 那又如何
會與你 共同渡過
都不枉過
瘋戀多 錯誤更多
如能從新做過
我會說 願能為你
提前做錯

[Pre-Chorus]
誰比你重要？
成功了敗了也完全無重要
誰比你重要？
狂風與暴雨都因你燃燒

[Chorus]
一追再追
只想追趕生命裡一分一秒 (一分一秒)
原來多麼可笑
你是真正目標
一追再追
追蹤一些生活最基本需要 (基本需要)
原來早不缺少 Woah
有了你 即使平凡卻最重要

[Bridge]

[C


In [5]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(sample):
    out = tokenizer(
        sample["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    out["labels"] = out["input_ids"].copy()
    return out

raw_dataset = Dataset.from_dict({"text": samples})
tokenized_dataset = raw_dataset.map(tokenize, remove_columns=["text"])
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.05, seed=42)
print(tokenized_dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
[transformers] Missing validation function in 'RotaryEmbeddingConfigMixin' for 'rope_type'='su'


Map:   0%|          | 0/583 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 553
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 30
    })
})


## 4. Load Pretrained InternVL2-4B Model (4-bit QLoRA)

In [ ]:
import torch
from transformers import AutoModel, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModel.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)
model.config.use_cache = False
print("Model loaded:", MODEL_ID)
print("Device:", next(model.parameters()).device)


[transformers] Missing validation function in 'RotaryEmbeddingConfigMixin' for 'rope_type'='su'
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


[transformers] Phi3ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


AttributeError: 'list' object has no attribute 'keys'

## 5. Configure LoRA Fine-Tuning Parameters

In [ ]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments

# ── Use the LLM backbone only ────────────────────────────────────────────
# InternVLChatModel.forward() has a custom VL signature that Trainer can't use.
# model.language_model is the underlying InternLM2 CausalLM — standard HF API.
lm = model.language_model
lm.config.use_cache = False

lm = prepare_model_for_kbit_training(
    lm,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

lm = get_peft_model(lm, lora_config)
lm.print_trainable_parameters()

OUTPUT_DIR = "/content/drive/MyDrive/cantopop/internvl2-4b-cantopop-lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    dataloader_pin_memory=False,
)
print("Training args configured.")


trainable params: 8,912,896 || all params: 3,829,722,112 || trainable%: 0.2327
Training args configured.


## 6. Define Training Loop & Start Fine-Tuning

In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=lm,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
)

print(f"Training on {len(tokenized_dataset['train'])} samples, "
      f"evaluating on {len(tokenized_dataset['test'])} samples.")
trainer.train()


Training on 553 samples, evaluating on 30 samples.


Epoch,Training Loss,Validation Loss
0,1.408800,1.378241


config.json: 0.00B [00:00, ?B/s]

ValueError: `rope_scaling`'s type field must be one of ['su', 'yarn'], got longrope

## 7. Save & Push Fine-Tuned Adapter to Hugging Face Hub

The LoRA adapter is first saved locally, then pushed to your HF Hub repo.
In `mm_direct_gen.py`, the app will load it directly by Hub ID — no file path management needed.

> Set `HF_REPO_ID` to `"your-hf-username/internvl2-4b-cantopop-lora"` before running.

In [ ]:
import getpass
from huggingface_hub import login

# ── Step 1: Securely enter your HF token (hidden input, not stored) ───────
HF_TOKEN = getpass.getpass("Enter your HuggingFace write token: ")
login(token=HF_TOKEN)

# ── Step 2: Set your Hub repo ID ──────────────────────────────────────────
HF_REPO_ID = "jeffreycheng234/internvl2-4b-cantopop-lora"

# ── Step 3: Save LoRA adapter (language model backbone only) locally ───────
LOCAL_ADAPTER_DIR = "/content/internvl2-4b-cantopop-lora"
lm.save_pretrained(LOCAL_ADAPTER_DIR)
tokenizer.save_pretrained(LOCAL_ADAPTER_DIR)
print(f"Adapter saved locally to: {LOCAL_ADAPTER_DIR}")

# ── Step 4: Push adapter + tokenizer to HF Hub ────────────────────────────
lm.push_to_hub(HF_REPO_ID, private=True)
tokenizer.push_to_hub(HF_REPO_ID, private=True)
print(f"Adapter pushed to HF Hub: https://huggingface.co/{HF_REPO_ID}")
print()
print("In mm_direct_gen.py, set INTERNVL_ADAPTER_ID to:")
print(f'  "{HF_REPO_ID}"')
